In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

# Read from Bronze
df = spark.table("bronze.aapl_stock_data")

In [0]:
# --- Schema enforcement ---
df = df.withColumn("open", F.col("open").cast(DoubleType())) \
       .withColumn("high", F.col("high").cast(DoubleType())) \
       .withColumn("low", F.col("low").cast(DoubleType())) \
       .withColumn("close", F.col("close").cast(DoubleType())) \
       .withColumn("volume", F.col("volume").cast(DoubleType())) \
       .withColumn("date", F.col("date").cast(DateType()))

In [0]:
# --- Data quality checks ---
# These mirror what you built in your DWH project
null_count = df.filter(F.col("close").isNull()).count()
duplicate_count = df.count() - df.dropDuplicates(["date", "ticker"]).count()

assert null_count == 0, f"Null check failed: {null_count} nulls in close"
assert duplicate_count == 0, f"Duplicate check failed: {duplicate_count} duplicates"

print(f"Quality checks passed — {df.count()} rows")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
# --- Drop unnecessary columns, add silver metadata ---
df_silver = df.withColumn("processed_at", F.current_timestamp())

# --- Write to Silver ---
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.aapl_stock_data")